# Reproducing Shaer on a Colab A100 (experiments E0 - E3)

**Notebook version 1.13.0** · updated 2026-11-20 22:17 UTC. Section 2 prints the version of the code it downloaded and warns if the two differ.

Branch `dev` of the fork (`thebazshah/Shaer`), forked from the official repository at commit `bba73cb`.
Everything this notebook produces is written to **your Google Drive** (`shaer-repro-<model>` folder).

Every cell is plain Python that calls a function of [`colab/shaer_colab.py`](https://github.com/thebazshah/Shaer/blob/dev/colab/shaer_colab.py)
(only the package installation in section 3 uses `!` commands). The module runs the repository's scripts in their own isolated Python
environment (next to Colab's own) and streams their output into the cell.

**You never have to wait blindly.** Every long step shows progress bars (downloads, loading, tokenising, generation, scoring, training),
prints a heartbeat line every ~20 s (time running, time since the last output, GPU use), and a step that is both silent and idle
(GPU below 10 % and almost no CPU) for `STALL_MINUTES` is stopped with a message telling you how to resume. Training runs in the
background; `sc.watch_training(cfg)` is a live view with a bar, loss, ETA and GPU, and can restart a hung run from its last checkpoint.

| Step | What | Needs | Roughly |
|---|---|---|---|
| E0 | Re-score the authors' released poems with the repo's evaluator | any runtime (CPU is fine) | minutes |
| E1 | Generate the 3,481 test poems with the released adapter, score them | GPU, 24 GB+ | hours |
| E2 | Same for the untuned base model | GPU, 24 GB+ | hours |
| E3 | Re-train Shaer (QLoRA, 4,236 steps), then generate and score | A100 40/80 GB class | the authors' run took 12.5 h |

**Runtime:** `Runtime > Change runtime type > A100 GPU`. The isolated environment uses the repository's pinned versions
(transformers 4.43.0, peft 0.12.0, bitsandbytes 0.43.3, torch 2.3.1), so Colab's own preinstalled PyTorch / transformers versions do not matter.

**GPU memory: a 40 GB A100 is enough.** Generation needs the 14 GB bf16 model plus a KV cache whose batch size adapts to the free memory
(25-32 prompts per batch on 40 GB). Training (4-bit QLoRA, batch 1, 1,024 tokens, gradient checkpointing on) needs roughly 12-14 GB by
my estimate; step 7a measures and prints the real peak.

**Before you start:** create a Hugging Face **read** token and add it as a Colab secret named `HF_TOKEN` (key icon in the
left sidebar, "Notebook access" on). The paper's base model
[`Navid-AI/Yehia-7B-preview`](https://huggingface.co/Navid-AI/Yehia-7B-preview) is gated; request access with that account.
**If access is not granted**, set `BASE_MODEL = "ALLaM-AI/ALLaM-7B-Instruct-preview"` in the first cell: it is the model
Yehia was fine-tuned from (same Llama architecture and a byte-identical tokenizer, Apache-2.0, not gated). That is a
documented deviation: E1 is skipped (the released adapter was trained on Yehia's weights), and E2/E3 then measure ALLaM, not Yehia.

**If the session disconnects:** run sections 1 to 3 again (they rebuild `/content`), then re-run the step you were in.
Generation resumes from the JSONL on Drive; training resumes from the newest complete checkpoint on Drive (keep `RUN_NAME` unchanged).

## 1. Configuration, Google Drive, secrets

In [1]:
import getpass
from google.colab import drive, userdata

NOTEBOOK_VERSION = "1.13.0"   # updated 2026-11-20 22:17 UTC; section 2 checks it against the downloaded code

# ---- edit these -------------------------------------------------------------------------------------------
REPO_URL   = "https://github.com/thebazshah/Shaer.git"
BRANCH     = "dev"
# Base model - exactly one of the two supported choices:
#   "Navid-AI/Yehia-7B-preview"            the paper's model (gated: needs the authors' approval)  -> PAPER FLOW
#   "ALLaM-AI/ALLaM-7B-Instruct-preview"   Yehia's public parent (Apache-2.0, not gated)            -> SUBSTITUTE FLOW
BASE_MODEL = "Navid-AI/Yehia-7B-preview"
FORCE_E1 = False       # substitute flow only: run the released adapter on ALLaM anyway (exploratory, not the paper's E1)
RUN_NAME   = "paper_repro_a100"                      # keep identical across restarts => automatic resume
NUM_SAMPLES = 1        # poems per test prompt in E1/E2/E3. The paper reports one; 3-5 lets you also measure best-of-k.
GEN_BATCH_SIZE = "auto"   # or a number. "auto" sizes the batch from free GPU memory (about 7-10 on an L4, 25-32 on a 40 GB A100, 32 on 80 GB)
STALL_MINUTES = 15     # a step that is silent AND idle (GPU < 10 %, almost no CPU) for this long is stopped, with instructions to resume
training_only = True  # True: only the main training (7b) runs; the cells of E0, E1, E2, 7a, the stop cell, 7c, E4 and the results
                      # table skip themselves. Set False to run the whole notebook.
# -----------------------------------------------------------------------------------------------------------

drive.mount("/content/drive")

def secret(name, prompt=None):
    try:
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return getpass.getpass(prompt) if prompt else ""

HF_TOKEN = secret("HF_TOKEN", "Hugging Face READ token (hidden): ")
GITHUB_TOKEN = secret("GITHUB_TOKEN")      # only needed if the fork is private (optional secret)
OPENROUTER_API_KEY = secret("OPENROUTER_API_KEY")   # only for the paid LLM judge (E4); add it as a Colab secret when you want that step
print("HF token set:", bool(HF_TOKEN), "| OpenRouter key set (needed only for E4):", bool(OPENROUTER_API_KEY))

Mounted at /content/drive
Hugging Face READ token (hidden): ··········
HF token set: True | OpenRouter key set (needed only for E4): False


## 2. Get the code from GitHub (branch `dev`) and point its output folders at Drive

In [2]:
import importlib, pathlib, subprocess, sys

# bootstrap: fetch the repository, then everything else is a call into colab/shaer_colab.py
REPO = pathlib.Path("/content/Shaer")
git_url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q", "--ff-only", git_url, BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, git_url, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)   # keep no token in .git/config

sys.path.insert(0, str(REPO / "colab"))
import shaer_colab as sc
importlib.reload(sc)                       # pick up the latest version if the module was imported before a pull

cfg = sc.make_config(base_model=BASE_MODEL, run_name=RUN_NAME, num_samples=NUM_SAMPLES, gen_batch_size=GEN_BATCH_SIZE,
                     force_e1=FORCE_E1, stall_minutes=STALL_MINUTES, repo_url=REPO_URL, branch=BRANCH, hf_token=HF_TOKEN,
                     openrouter_key=OPENROUTER_API_KEY)
sc.sync_repo(cfg)                          # training checkpoints and all experiment outputs live on Drive
sc.check_version(NOTEBOOK_VERSION)         # warns if this notebook copy and the downloaded code differ

PAPER FLOW: Yehia-7B-preview. E0, E1, E2, E3 run as in the paper.
base model: Navid-AI/Yehia-7B-preview | Drive folder: /content/drive/MyDrive/shaer-repro-yehia | a silent, idle step is stopped after 15 min
code: 3eaafb7 Add the E4 judge section back after the notebook was saved again | outputs -> /content/drive/MyDrive/shaer-repro-yehia


## 3. Training environment (about 5 minutes on a fresh runtime) and GPU check

This notebook runs on Colab's **latest runtime** (2026.10 or newer: Ubuntu 24.04, Python 3.13, CUDA 13), not pinned to an old one,
so a runtime starts quickly. Training does not use Colab's own packages (torch 2.11, transformers 5.x): it runs in an isolated
environment with the paper's exact versions (Python 3.12, torch 2.3.1 with its own CUDA 12.1, transformers 4.43.0, peft 0.12.0,
bitsandbytes 0.43.3), so the results do not depend on what Colab ships, and the newer GPU driver runs these older builds.
On a runtime without Python 3.12 (2026.10 has 3.13), `uv` downloads a standalone Python 3.12 first (well under a minute).

The first cell installs into the isolated environment `/content/venv` with the repository's pinned versions
([`requirements-colab.txt`](https://github.com/thebazshah/Shaer/blob/dev/colab/requirements-colab.txt)). It is safe to run again: when
everything is already installed it finishes in seconds, and it repairs an environment left over from an older version of this notebook.
Its output shows the installer's own progress.

The GPU check refuses to continue training on a GPU this stack cannot use faithfully: no bf16 (T4), or a Blackwell
card (compute capability 12.x), for which the pinned torch 2.3.1 / bitsandbytes 0.43.3 have no kernels.

The check prints the Colab runtime (release, OS, the notebook's Python, GPU driver), the training environment's Python and every
pinned package with its installed version, and flags any that differ from `requirements-colab.txt`.

In [ ]:
!pip -q install uv
!uv venv --python 3.12 --python-preference managed --allow-existing /content/venv   # uv fetches Python 3.12 when the runtime has another
!uv pip install --python /content/venv/bin/python torch==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install --python /content/venv/bin/python -r /content/Shaer/colab/requirements-colab.txt
!uv pip check --python /content/venv/bin/python

In [4]:
sc.check_environment(cfg)


▶ checking the training environment and the GPU   (expect a few seconds)
torch 2.3.1+cu121 | transformers 4.43.0 | peft 0.12.0 | bitsandbytes 0.43.3 | protobuf 4.25.3
GPU: NVIDIA L4 | 22 GB | compute capability 8.9 | bf16 supported: True
E0 (scoring)    : fine on any runtime
E1/E2 (generate): OK
E3 (training)   : NOT RECOMMENDED on this GPU (needs bf16, older than Blackwell, a 40 GB class card)
++++++++++++++++++++++ DEBUG INFO END ++++++++++++++++++++++
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Checking that the library is importable and CUDA is callable...
SUCCESS!
Installation was successful!
✔ environment check: done in 00:12


In [5]:
sc.check_hf_access(cfg)       # fails early if the base model or its tokenizer cannot be loaded


▶ checking access to Navid-AI/Yehia-7B-preview and loading its tokenizer   (expect under a minute)
access to Navid-AI/Yehia-7B-preview : OK
slow tokenizer loads OK | vocab 64000 | LlamaTokenizer
✔ access check: done in 00:06


## 4. E0 - re-score the authors' released poems

Uses the repository's own evaluator, unchanged, on the CPU (deterministic). On the 3,481 released poems it reproduces
the paper's table: meter conformity 0.9064, base-meter accuracy 95.17 %, exact count 83.40 %, count adherence 0.9792.

In [6]:
if not training_only:
    sc.run_e0(cfg)
else:
    print("skipped: training_only = True (set it to False in section 1 to run this cell)")


▶ E0: exporting the authors' released poems and re-scoring them   (expect about 3-5 minutes)
Repo card metadata block was not found. Setting CardData to empty.
  Generating test split          [░░░░░░░░░░░░░░░░░░░░]   0%  0/3481  elapsed 00:00
  Generating test split          [████████████████████] 100%  3481/3481  elapsed 00:00  ETA 00:00
wrote 3481 rows -> runs/e0/generations.jsonl

▶ scoring 3,481 poems with the repository's evaluator (CPU, deterministic)   (expect a few minutes)
  … scoring E0  [█░░░░░░░░░░░░░░░░░░░]   5.2%  182/3,481  elapsed 00:10
  … scoring E0  [███░░░░░░░░░░░░░░░░░]  15.3%  532/3,481  elapsed 00:20
  … scoring E0  [█████░░░░░░░░░░░░░░░]  25.0%  870/3,481  elapsed 00:30
  … scoring E0  [███████░░░░░░░░░░░░░]  34.0%  1,183/3,481  elapsed 00:40
  … scoring E0  [████████░░░░░░░░░░░░]  41.6%  1,447/3,481  elapsed 00:50
  … scoring E0  [██████████░░░░░░░░░░]  51.2%  1,784/3,481  elapsed 01:00
  … scoring E0  [████████████░░░░░░░░]  60.2%  2,096/3,481  elapsed 01:10

## 5. E1 - released adapter (`Shaer-AI/Shaer-adapters`, `adapters/fresh_sft/train/best`)

Decoding follows the paper's Table 15 (T 0.55, top-p 0.90, repetition penalty 1.08, 768 new tokens). The base model is
loaded in bf16 without quantisation (the adapter was trained on an NF4 base; the generation script's `--quantize nf4` matches that instead).

In [7]:
if not training_only:
    sc.run_generation(cfg, "e1")
else:
    print("skipped: training_only = True (set it to False in section 1 to run this cell)")


▶ E1: generating the test poems with the released Shaer adapter on Yehia-7B-preview   (expect model download + load first (about 5-10 min on a fresh runtime), then the progress bar with an ETA)
[1/5] loading the test prompts ...
  Generating train split         [░░░░░░░░░░░░░░░░░░░░]   0%  0/109070  elapsed 00:00
  Generating train split         [████░░░░░░░░░░░░░░░░]  18%  20000/109070  elapsed 00:00  ETA 00:00
  Generating train split         [██████░░░░░░░░░░░░░░]  29%  32000/109070  elapsed 00:00  ETA 00:00
  Generating train split         [████████░░░░░░░░░░░░]  40%  44000/109070  elapsed 00:00  ETA 00:00
  Generating train split         [██████████░░░░░░░░░░]  51%  55535/109070  elapsed 00:00  ETA 00:00
  Generating train split         [█████████████░░░░░░░]  63%  68535/109070  elapsed 00:00  ETA 00:00
  Generating train split         [████████████████░░░░]  78%  84535/109070  elapsed 00:00  ETA 00:00
  Generating train split         [██████████████████░░]  92%  100535/109070  e

## 6. E2 - untuned base model (same prompts, no adapter)

In [8]:
if not training_only:
    sc.run_generation(cfg, "e2")
else:
    print("skipped: training_only = True (set it to False in section 1 to run this cell)")


▶ E2: generating the test poems with the untuned Navid-AI/Yehia-7B-preview   (expect model download + load first (about 5-10 min on a fresh runtime), then the progress bar with an ETA)
[1/5] loading the test prompts ...
[2/5] loading the tokenizer ...
[3/5] loading the base model Navid-AI/Yehia-7B-preview in bfloat16; the first time this downloads about 14 GB ...
  Loading checkpoint shards      [░░░░░░░░░░░░░░░░░░░░]   0%  0/3  elapsed 00:00
  Loading checkpoint shards      [███████░░░░░░░░░░░░░]  33%  1/3  elapsed 00:01  ETA 00:03
  Loading checkpoint shards      [█████████████░░░░░░░]  67%  2/3  elapsed 00:03  ETA 00:01
  Loading checkpoint shards      [████████████████████] 100%  3/3  elapsed 00:04  ETA 00:00
[4/5] no adapter: generating with the untuned model
[5/5] tokenising the prompts and building batches ...
batch size: 11 (auto), longest prompt 329 tokens
generating 0 of 3481 poems (3481 already done) in 317 batches per sample; each batch takes up to a few minutes (up to 768

## 7. E3 - re-train

**7a. Smoke test first (about 10 minutes, includes the 15 GB model download).** It trains 6 steps with the real 4-bit
configuration, which proves bitsandbytes, the data path and the checkpointing work on this runtime before you commit hours to it,
and prints the measured peak GPU memory.

In [9]:
if not training_only:
    sc.train_smoke_test(cfg)
else:
    print("skipped: training_only = True (set it to False in section 1 to run this cell)")


▶ smoke test: 6 training steps with the real 4-bit setup   (expect about 10-15 min, mostly the 15 GB model download and loading)
2026-11-17 22:08:55,498 | INFO | Starting SFT | mode=sanity run_name=sanity_paper_repro_a100
2026-11-17 22:08:56,134 | INFO | Base model is the paper's: Navid-AI/Yehia-7B-preview
2026-11-17 22:08:56,135 | INFO | Output dir: /content/drive/MyDrive/shaer-repro-yehia/sft_outputs/sanity_check/sanity_paper_repro_a100
2026-11-17 22:08:57,842 | INFO | push_to_hub is off: nothing will be created or uploaded on the Hub (HF_TOKEN is only used to download gated files)
2026-11-17 22:09:01,432 | INFO | Resume decision: {'timestamp_utc': '2026-11-17T22:08:58.453002+00:00', 'mode': 'sanity', 'resume_mode': 'auto', 'result': 'resume', 'reason': 'auto_local_checkpoint', 'checkpoint_repo_path': None, 'local_resume_path': '/content/drive/MyDrive/shaer-repro-yehia/sft_outputs/sanity_check/sanity_paper_repro_a100/checkpoint-6'}
  Generating train split         [░░░░░░░░░░░░░░░░░

**7b. Full run.** Trains right here in this cell and prints its progress as it happens: the step bar with ETA (a new line about every
30 s), the loss every 10 steps and each evaluation. Everything is also appended to the log on Drive. While the run prints nothing (model
loading, tokenising and packing the data, writing a checkpoint), a heartbeat line every 20 s shows elapsed time, GPU and CPU use.
Keep the browser tab open. Stopping the cell stops training cleanly; a disconnected or recycled runtime does too. Either way, re-run
sections 1-3 and this cell: it resumes from the newest complete checkpoint (a half-written one is skipped automatically) and skips the step-0 evaluation it already did. Besides the regular checkpoint every 200 steps (the paper's evaluation schedule), a resume checkpoint is written every **20 minutes** of training on an A100 / H100 and every **10 minutes** on other GPUs (`save_minutes="auto"`, the default; a number sets it), so at most that much training is lost. When training stops (stop button), finishes or fails, the cell **flushes Drive** by itself (`sc.flush_drive()`): it uploads the newest checkpoint before you can end the session, then mounts Drive again. Wait for `Drive flushed` before ending or switching the session. Each is about 1 GB written to Drive. Only two checkpoints are kept: the **latest** (to resume from) and the **best** (lowest eval loss so far, used in 7c), or one when they are the same; the previous latest is deleted only after the new one is completely written. `save_minutes=None` turns it off, `save_steps=50` adds a step-based one as well; neither changes the evaluation or the results.
Gradient checkpointing is **off** in the call below: it saves memory by recomputing every layer's forward pass during the backward
pass, roughly 20-30 % of each step. Without it the run uses more GPU memory (with it on, an L4 showed 7.3 of 22.5 GB in use; my estimate
without it is about 15-20 GB, not measured) and the maths is the same. If the GPU runs out of memory, the run restarts by itself from the
last checkpoint with it switched back on. Switching it on or off does not stop a run from resuming. Use `gradient_checkpointing=True`
to keep it on from the start.

The step line's speed and ETA are measured by the notebook from the last updates (tqdm's own ETA is far off right after a resume,
because the bar jumps to the resumed step at once). Evaluation, every 200 steps over the 3,481 eval rows, runs in batches of **32** on an 80 GB GPU, 16 on 40 GB and 8 below
(`eval_batch_size="auto"`; the paper's configuration uses 1, which took ~31 min per evaluation on an L4). Training is not affected; the
reported eval loss shifts slightly, since a batch's loss is averaged over all its tokens rather than per row. After an out-of-memory
error during evaluation the run restarts with half the batch size. `eval_batch_size=1` keeps the paper's setting.

In [10]:
sc.checkpoint_report(cfg)     # what this run will resume from (checkpoints on Drive, complete or not)
sc.start_training(cfg, gradient_checkpointing=False)
# live progress in this cell; gradient checkpointing off (faster, same maths; back on by itself after an out-of-memory error);
# evaluation in batches of 32 on an 80 GB GPU, 16 on 40 GB, 8 below (eval_batch_size="auto"); resume checkpoint every 20 min on an A100 / H100, 10 min on other GPUs, plus every 200 steps;
# a run that is silent AND idle for STALL_MINUTES restarts from the last checkpoint; Drive is flushed when it stops or finishes.
# sc.start_training(cfg, background=True) for the old detached run, followed with sc.watch_training(cfg)

▶ training started (pid 9915)
  log: /content/drive/MyDrive/shaer-repro-yehia/logs/train_paper_repro_a100.log
  follow it live with: sc.watch_training(cfg)


Only for a run started with `sc.start_training(cfg, background=True)`: a live view of the detached process (bar, loss, ETA, GPU and
CPU use), refreshed every 20 s. The default run above prints its own progress, so you do not need this cell. If a background run is
still going (for example one started before this change), `sc.start_training(cfg)` says so: stop it with `sc.stop_training(cfg)` first.
`sc.training_status(cfg)` is the one-shot version.

In [11]:
if False:   # only for sc.start_training(cfg, background=True); the default run above already shows its progress
    sc.watch_training(cfg, auto_restart=True)

training: RUNNING | GPU 100% busy, 9.5/22.5 GB
  [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   2.5%  step 104/4,236  ETA 47:10:22  (41.10s/it)
  train loss 2.5081 | lr 6.25e-05 | epoch 0.05 | last eval loss 3.0791
  log last updated 00:09 ago
stopped watching; training continues in the background (sc.watch_training(cfg) to look again)


**Before you end or switch a Colab session:** 7b flushes Drive by itself when it stops or finishes (wait for `Drive flushed`).
If that was interrupted, or you wrote to Drive in some other way, run the cell below: it uploads everything still pending in Colab's
Drive cache (a checkpoint is about 1 GB) and mounts Drive again. Ending a session before the upload finishes loses the newest
checkpoint, and the next session cannot resume from it.

In [ ]:
sc.flush_drive()               # only needed if 7b's own flush was interrupted; harmless to run again

In [13]:
if not training_only:
    cfg_allam = sc.make_config(base_model="ALLaM-AI/ALLaM-7B-Instruct-preview", run_name=RUN_NAME,
                               repo_url=REPO_URL, branch=BRANCH, hf_token=HF_TOKEN)
    sc.training_status(cfg_allam)   # look first: running? which step? which checkpoints exist?
    sc.stop_training(cfg_allam)     # asks it to stop; force-kills it if it hasn't exited after ~20 s
else:
    print("skipped: training_only = True (set it to False in section 1 to run this cell)")

SUBSTITUTE FLOW: ALLaM-7B-Instruct-preview. E1 is skipped (the released adapter was trained on Yehia). E2/E3 are NOT the paper's Yehia numbers.
base model: ALLaM-AI/ALLaM-7B-Instruct-preview | Drive folder: /content/drive/MyDrive/shaer-repro-allam | a silent, idle step is stopped after 15 min
training: NOT running | GPU 100% busy, 9.5/22.5 GB
  [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   4.6%  step 196/4,236  ETA 21:04:20  (18.78s/it)
  train loss 2.1054 | lr 8.00e-05 | epoch 0.09 | last eval loss 2.3817
  log last updated 4:06:42 ago
  checkpoints: []
training is not running


**7c. After training finished:** generate with the best checkpoint (lowest eval loss), score, summarise.

In [ ]:
if not training_only:
    sc.run_generation(cfg, "e3")
else:
    print("skipped: training_only = True (set it to False in section 1 to run this cell)")

## 8. E4 - LLM judge (optional, paid)

Scores the poems of a generation step on five metrics (description adherence, meaning, fluency, coherence, poeticness; 1-5) with the
repository's own judge: the paper's judge model `qwen/qwen3-235b-a22b-2507`, the strict v2 prompts and temperature 0, through
[OpenRouter](https://openrouter.ai). **This is a paid API: one poem is five calls**, so all 3,481 poems are 17,405 calls. Only the
poem and its description are sent, never the reference poem. By default only sample 0 of each prompt is judged.

**Setup:** add your OpenRouter key as a Colab secret named `OPENROUTER_API_KEY` and run the configuration cell (section 1) again. If you
installed the environment before this step existed, run the install cell in section 3 again (it adds `curl_cffi`).

**Order:** the preview shows the plan and the exact prompt without any call; the trial judges 10 poems (50 calls) and checks the key and the
model; then the full run, with a progress bar, rows per minute, ETA and the API-reported cost. `max_cost_usd=...` stops at a budget.
Everything is safe to stop and run again: finished poems are kept, failed ones are judged again, and a hung API is detected and stopped.
The scores can be compared with the paper's only when the paper's judge model is used (the results table says so).

In [ ]:
if not training_only:
    sc.run_judge(cfg, "e3", dry_run=True)    # the plan and the exact prompt; no API call, no cost. Use "e1" or "e2" for the other steps
else:
    print("skipped: training_only = True (set it to False in section 1 to run this cell)")

In [ ]:
if not training_only:
    sc.run_judge(cfg, "e3", limit_rows=10)   # trial: 10 poems x 5 metrics = 50 calls; checks the key, the model and the credits
else:
    print("skipped: training_only = True (set it to False in section 1 to run this cell)")

In [ ]:
if not training_only:
    sc.run_judge(cfg, "e3")                  # all poems; e.g. sc.run_judge(cfg, "e3", max_cost_usd=20) to stop at a budget
else:
    print("skipped: training_only = True (set it to False in section 1 to run this cell)")

## 9. Results: your runs against the paper (saved to Drive as `results.md`)

In [ ]:
if not training_only:
    table = sc.results_table(cfg)
else:
    print("skipped: training_only = True (set it to False in section 1 to run this cell)")

In [ ]:
sc.plot_training_curves(cfg)

## 10. Release the runtime

With **Runtime > Run all**, this last cell runs once everything above has finished (for example the full training run in 7b). It
uploads anything still pending to Drive, then releases the Colab runtime, so the GPU stops using compute units while you are away.
Stopping a cell or an error stops *Run all* before it gets here, so the runtime is only released after a normal finish. All
outputs are on Drive; the next session starts with sections 1-3.

In [ ]:
sc.flush_drive(remount=False)   # make sure every checkpoint and result has reached Drive first
from google.colab import runtime
runtime.unassign()              # ends this Colab session and releases the GPU